<a href="https://colab.research.google.com/github/RafaelaMlucca/conformal-violence-prediction/blob/main/03_locart_aplicacao_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 — Locart adaptado para classificação binária

Implementa Locart (Cabezas et al., 2024) adaptado para classificação binária
seguindo Frohlich et al. (2025) e aplica aos três desfechos.

**Mudança:** mantém matrizes esparsas. DecisionTreeRegressor aceita esparso direto.

## 1. Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import math
import json
from scipy.sparse import load_npz
from sklearn.tree import DecisionTreeRegressor

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')
SAIDA = DRIVE / 'conformal_violence'

RANDOM_STATE = 42
ALPHA = 0.1

## 2. Carregando dados e modelos

In [4]:
X_train = load_npz(SAIDA / 'X_train.npz')
X_cal   = load_npz(SAIDA / 'X_cal.npz')
X_test  = load_npz(SAIDA / 'X_test.npz')

y_train = pd.read_parquet(SAIDA / 'y_train.parquet')
y_cal   = pd.read_parquet(SAIDA / 'y_cal.parquet')
y_test  = pd.read_parquet(SAIDA / 'y_test.parquet')

with open(SAIDA / 'metadados.json') as f:
    meta = json.load(f)
ALVOS = meta['alvos']
feature_names = meta['feature_names']

with open(SAIDA / 'resultados_baseline.pkl', 'rb') as f:
    baseline = pickle.load(f)
modelos = baseline['modelos']

Tudo carregado.


## 3. Função aprender_locart

In [5]:
def aprender_locart(modelo, X_cal, y_cal_alvo, alpha=ALPHA,
                    max_depth=5, min_samples_leaf=None, random_state=RANDOM_STATE):
    # passo 1: resíduos
    proba_cal = modelo.predict_proba(X_cal)
    residuos = 1 - proba_cal[np.arange(len(y_cal_alvo)), y_cal_alvo]

    # passo 2: árvore — aceita esparso direto
    if min_samples_leaf is None:
        min_samples_leaf = max(50, int(0.005 * X_cal.shape[0]))

    arvore = DecisionTreeRegressor(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state,
    )
    arvore.fit(X_cal, residuos)

    # passo 3: mapa de calibração -> folha
    folhas_cal = arvore.apply(X_cal)

    # passo 4: quantil por folha
    quantis = {}
    for folha in np.unique(folhas_cal):
        mascara = folhas_cal == folha
        probs_folha = residuos[mascara]
        n = len(probs_folha)
        if n == 0:
            continue
        cobertura_ajustada = math.ceil((n + 1) * (1 - alpha)) / n
        cobertura_ajustada = min(cobertura_ajustada, 1.0)
        q_residuo = np.quantile(probs_folha, cobertura_ajustada, method='higher')
        quantis[int(folha)] = 1.0 - q_residuo

    return {
        'arvore': arvore,
        'quantis': quantis,
        'min_samples_leaf': min_samples_leaf,
        'n_folhas': arvore.get_n_leaves(),
    }

## 4. Função predizer_locart

In [6]:
def predizer_locart(modelo, locart, X_test):
    arvore = locart['arvore']
    quantis = locart['quantis']

    proba_test = modelo.predict_proba(X_test)
    folhas_test = arvore.apply(X_test)

    n_test = X_test.shape[0]
    conjuntos = np.zeros((n_test, 2), dtype=bool)

    for i in range(n_test):
        folha = int(folhas_test[i])
        q = quantis.get(folha, 0.5)
        for classe in [0, 1]:
            if proba_test[i, classe] >= q:
                conjuntos[i, classe] = True

    return conjuntos, folhas_test

## 5. Função auxiliar de avaliação

In [7]:
def avaliar_conjuntos(y_verdadeiro, conjuntos):
    n = len(y_verdadeiro)
    cobertos = np.array([conjuntos[i, y_verdadeiro[i]] for i in range(n)])
    tamanhos = conjuntos.sum(axis=1)
    return {
        'cobertura': float(cobertos.mean()),
        'tamanho_medio': float(tamanhos.mean()),
        'pct_vazios': float((tamanhos == 0).mean()),
        'pct_unitarios': float((tamanhos == 1).mean()),
        'pct_duplos': float((tamanhos == 2).mean()),
    }

## 6. Rodando Locart para os 3 desfechos

In [8]:
resultados_locart = {}

for alvo in ALVOS:
    modelo = modelos[alvo]
    y_cal_alvo = y_cal[alvo].values
    y_test_alvo = y_test[alvo].values

    locart = aprender_locart(modelo, X_cal, y_cal_alvo)
    conjuntos, folhas_test = predizer_locart(modelo, locart, X_test)
    metricas = avaliar_conjuntos(y_test_alvo, conjuntos)

    resultados_locart[alvo] = {
        **metricas,
        'conjuntos': conjuntos,
        'folhas_test': folhas_test,
        'locart': locart,
    }

    print(f"{alvo}: {locart['n_folhas']} folhas | "
          f"cob={metricas['cobertura']:.4f} | "
          f"tam={metricas['tamanho_medio']:.4f} | "
          f"vazios={metricas['pct_vazios']:.1%}")

y_fisic: 28 folhas | cob=0.9036 | tam=1.1824 | vazios=2.2%
y_psico: 28 folhas | cob=0.9005 | tam=1.2302 | vazios=3.4%
y_sexu: 29 folhas | cob=0.8910 | tam=1.0003 | vazios=6.3%


## 7. Cobertura por folha (cobertura local)

In [9]:
def cobertura_por_folha(y_test_alvo, conjuntos, folhas_test):
    folhas_unicas = np.unique(folhas_test)
    linhas = []
    for f in folhas_unicas:
        mask = folhas_test == f
        if mask.sum() == 0:
            continue
        y_f = y_test_alvo[mask]
        c_f = conjuntos[mask]
        cobertos = np.array([c_f[i, y_f[i]] for i in range(len(y_f))])
        tamanhos = c_f.sum(axis=1)
        linhas.append({
            'folha': int(f),
            'n_teste': int(mask.sum()),
            'cobertura': float(cobertos.mean()),
            'tam_medio': float(tamanhos.mean()),
            'pct_unitarios': float((tamanhos == 1).mean()),
            'pct_vazios': float((tamanhos == 0).mean()),
            'pct_duplos': float((tamanhos == 2).mean()),
        })
    return pd.DataFrame(linhas).sort_values('n_teste', ascending=False)

for alvo in ALVOS:
    print(f'\n=== {alvo} ===')
    df = cobertura_por_folha(
        y_test[alvo].values,
        resultados_locart[alvo]['conjuntos'],
        resultados_locart[alvo]['folhas_test'],
    )
    resultados_locart[alvo]['tabela_folhas'] = df
    print(df.head(15).to_string(index=False))


=== y_fisic ===
 folha  n_teste  cobertura  tam_medio  pct_unitarios  pct_vazios  pct_duplos
    34   134237   0.920938   1.339564       0.660436    0.000000    0.339564
    29   133572   0.880207   0.993389       0.993389    0.006611    0.000000
    13   122430   0.910218   1.136347       0.863653    0.000000    0.136347
    26    75140   0.893585   0.904791       0.904791    0.095209    0.000000
     6    64010   0.905968   0.990392       0.990392    0.009608    0.000000
    28    51078   0.884745   0.894593       0.894593    0.105407    0.000000
     7    44256   0.910566   0.958401       0.958401    0.041599    0.000000
    37    32463   0.938977   1.592736       0.407264    0.000000    0.592736
    18    30308   0.889435   1.722417       0.277583    0.000000    0.722417
    25    25652   0.888391   0.915679       0.915679    0.084321    0.000000
    35    19419   0.918688   1.590710       0.409290    0.000000    0.590710
    38    13581   0.931301   1.514837       0.485163    0.0

## 8. Comparação com MAPIE

In [10]:
linhas = []
for alvo in ALVOS:
    lac = baseline['lac'][alvo]
    aps = baseline['aps'][alvo]
    loc = resultados_locart[alvo]
    for nome, res in [('LAC', lac), ('APS', aps), ('Locart', loc)]:
        linhas.append({
            'desfecho': alvo, 'metodo': nome,
            'cobertura': res['cobertura'],
            'tam_medio': res['tamanho_medio'],
            'pct_vazios': res['pct_vazios'],
            'pct_unitarios': res['pct_unitarios'],
            'pct_duplos': res['pct_duplos'],
        })

tabela_comparativa = pd.DataFrame(linhas)
print(tabela_comparativa.to_string(index=False))

desfecho metodo  cobertura  tam_medio  pct_vazios  pct_unitarios  pct_duplos
 y_fisic    LAC   0.906491   1.118269    0.000000       0.881731    0.118269
 y_fisic    APS   0.906491   1.118269    0.000000       0.881731    0.118269
 y_fisic Locart   0.903611   1.182381    0.022083       0.773452    0.204465
 y_psico    LAC   0.906176   1.154893    0.000000       0.845107    0.154893
 y_psico    APS   0.906176   1.154893    0.000000       0.845107    0.154893
 y_psico Locart   0.900529   1.230186    0.034387       0.701040    0.264573
  y_sexu    LAC   0.894666   0.950211    0.049789       0.950211    0.000000
  y_sexu    APS   0.894666   0.950211    0.049789       0.950211    0.000000
  y_sexu Locart   0.890969   1.000342    0.063201       0.873257    0.063542


## 9. Salvando

In [11]:
resultados_finais = {
    'locart': resultados_locart,
    'tabela_comparativa': tabela_comparativa,
    'alpha': ALPHA,
}

with open(SAIDA / 'resultados_locart.pkl', 'wb') as f:
    pickle.dump(resultados_finais, f)

print(f'Salvo em {SAIDA / "resultados_locart.pkl"}')

Salvo em /content/drive/MyDrive/projeto_violencia_mulher/conformal_violence/resultados_locart.pkl
